model building

In [2]:
import pandas as pd
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.combine import SMOTEENN
import joblib

In [3]:
dfm=pd.read_csv('churnencoded')
dfm.drop(columns='Unnamed: 0',inplace=True)
dfm.head()

,SeniorCitizen,MonthlyCharges,TotalCharges,Churn,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,...,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_grp_1-12,tenure_grp_13-24,tenure_grp_25-36,tenure_grp_37-48,tenure_grp_49-60,tenure_grp_61-72
0,0,29.85,29.85,0,1,0,0,1,1,0,...,0,0,1,0,1,0,0,0,0,0
1,0,56.95,1889.50,0,0,1,1,0,1,0,...,0,0,0,1,0,0,1,0,0,0
2,0,53.85,108.15,1,0,1,1,0,1,0,...,0,0,0,1,1,0,0,0,0,0
3,0,42.30,1840.75,0,0,1,1,0,1,0,...,1,0,0,0,0,0,0,1,0,0
4,0,70.70,151.65,1,1,0,1,0,1,0,...,0,0,1,0,1,0,0,0,0,0


In [15]:
y=dfm['Churn']
x=dfm.drop(columns='Churn')
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)
joblib.dump(x_train.columns,'model columns.pkl')

['model columns.pkl']

decision tree

In [5]:
model_dt=DecisionTreeClassifier(criterion='gini',random_state=100,max_depth=8)
model_dt.fit(x_train,y_train)
dt_pred=model_dt.predict(x_test)

In [6]:
print(classification_report(y_test,dt_pred))

              precision    recall  f1-score   support

           0       0.83      0.87      0.85      1030
           1       0.59      0.51      0.55       377

    accuracy                           0.78      1407
   macro avg       0.71      0.69      0.70      1407
weighted avg       0.77      0.78      0.77      1407



In [7]:
print(confusion_matrix(y_test,dt_pred))

[[899 131]
 [185 192]]


In [8]:
print(accuracy_score(y_test,dt_pred))

0.775408670931059


class imbalanced fix

In [9]:
sm=SMOTEENN(random_state=32)
x_res,y_res=sm.fit_resample(x,y)
xr_train,xr_test,yr_train,yr_test=train_test_split(x_res,y_res,test_size=0.2,random_state=32)
model_dtr=DecisionTreeClassifier(criterion='gini',random_state=45,max_depth=10)
model_dtr.fit(xr_train,yr_train)
yr_pred=model_dtr.predict(xr_test)
print(classification_report(yr_test,yr_pred))

              precision    recall  f1-score   support

           0       0.93      0.93      0.93       510
           1       0.95      0.94      0.95       662

    accuracy                           0.94      1172
   macro avg       0.94      0.94      0.94      1172
weighted avg       0.94      0.94      0.94      1172



new model(LOGROG)

In [10]:
logrog=LogisticRegression()
logrog.fit(xr_train,yr_train)
yr_predlogrog=logrog.predict(xr_test)
print(classification_report(yr_test,yr_predlogrog))

              precision    recall  f1-score   support

           0       0.91      0.91      0.91       510
           1       0.93      0.93      0.93       662

    accuracy                           0.92      1172
   macro avg       0.92      0.92      0.92      1172
weighted avg       0.92      0.92      0.92      1172



c:\Users\abysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


RandomForest

In [11]:
model_rf=RandomForestClassifier(random_state=33,n_estimators=100,criterion='gini')
model_rf.fit(xr_train,yr_train)
yr_predrf=model_rf.predict(x_test)
print(classification_report(y_test,yr_predrf))


              precision    recall  f1-score   support

           0       0.91      0.78      0.84      1030
           1       0.57      0.79      0.66       377

    accuracy                           0.79      1407
   macro avg       0.74      0.79      0.75      1407
weighted avg       0.82      0.79      0.79      1407



saving the model

In [12]:
import joblib
joblib.dump(model_dtr,"model redt")
model=joblib.load("model redt")
model.score(xr_test,yr_test)

0.939419795221843